## Scenario 3: Multiple data scientists working on multiple ML models

MLflow setup:
* Tracking server: yes, remote server (EC2).
* Backend store: postgresql database.
* Artifacts store: s3 bucket.

The experiments can be explored by accessing the remote server.

The example uses AWS to host a remote server. In order to run the example you'll need an AWS account. Follow the steps described in the file `mlflow_on_aws.md` to create a new AWS account and launch the tracking server. 

In [1]:
import mlflow
import os


TRACKING_SERVER_HOST = "ec2-3-27-11-132.ap-southeast-2.compute.amazonaws.com" # public DNS of the EC2 instance
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")

/home/codespace/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/codespace/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (
/home/codespace/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://ec2-3-27-11-132.ap-southeast-2.compute.amazonaws.com:5000'


In [3]:
mlflow.search_experiments() # list_experiments API has been removed, you can use search_experiments instead.()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-scenario3/2', creation_time=1773156458256, experiment_id='2', last_update_time=1773156458256, lifecycle_stage='active', name='my-experiment-2', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-scenario3/1', creation_time=1773153689684, experiment_id='1', last_update_time=1773153689684, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-scenario3/0', creation_time=1773153385469, experiment_id='0', last_update_time=1773153385469, lifecycle_stage='active', name='Default', tags={}>]

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from dotenv import load_dotenv

load_dotenv()

# You can now use os.getenv() to access them or let boto3 pick them up automatically
access_key = os.getenv("AWS_ACCESS_KEY_ID")
secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
region = os.getenv("AWS_DEFAULT_REGION")

mlflow.set_experiment("my-experiment-2")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, name="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/03/10 15:36:09 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpdxx4xmm2/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.6.0', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/10 15:36:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


default artifacts URI: 's3://mlflow-artifacts-remote-scenario3/2/70f1d07c8c104a869effa0c999da63a0/artifacts'
🏃 View run able-finch-127 at: http://ec2-3-27-11-132.ap-southeast-2.compute.amazonaws.com:5000/#/experiments/2/runs/70f1d07c8c104a869effa0c999da63a0
🧪 View experiment at: http://ec2-3-27-11-132.ap-southeast-2.compute.amazonaws.com:5000/#/experiments/2


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-scenario3/2', creation_time=1773156458256, experiment_id='2', last_update_time=1773156458256, lifecycle_stage='active', name='my-experiment-2', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-scenario3/1', creation_time=1773153689684, experiment_id='1', last_update_time=1773153689684, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-scenario3/0', creation_time=1773153385469, experiment_id='0', last_update_time=1773153385469, lifecycle_stage='active', name='Default', tags={}>]

### Interacting with the model registry

In [6]:
from mlflow.tracking import MlflowClient


client = MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")

In [7]:
client.search_registered_models()

[]

In [8]:
run_id = client.search_runs(experiment_ids=['2'])[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2026/03/10 15:37:10 WARNING mlflow.tracking._model_registry.fluent: Run with id 70f1d07c8c104a869effa0c999da63a0 has no artifacts at artifact path 'models', registering model based on models:/m-604935f3165e417b934e59d46ba84b33 instead
2026/03/10 15:37:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1773157030876, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1773157030876, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='70f1d07c8c104a869effa0c999da63a0', run_link='', source='models:/m-604935f3165e417b934e59d46ba84b33', status='READY', status_message=None, tags={}, user_id='', version='1'>